# Chapter 19 — Training and Deploying TensorFlow Models at Scale
Companion notebook to `part2_deep_learning/19_training_deploying_at_scale.md`.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras

# A small trained model to export (reuse Fashion MNIST from Chapter 10/11 if you have it in memory,
# otherwise train a quick one here)
fashion_mnist = keras.datasets.fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(X_train, y_train, epochs=3, validation_split=0.1)

## Export to SavedModel format

In [ ]:
model_version = "0001"
model_name = "my_mnist_model"
model_path = os.path.join(model_name, model_version)
tf.saved_model.save(model, model_path)
print("Saved to:", model_path)

# Inspect it
os.system(f"saved_model_cli show --dir {model_path} --all")

## Load it back two ways

In [ ]:
# 1. As a raw SavedModel
raw_model = tf.saved_model.load(model_path)
y_pred_raw = raw_model(tf.constant(X_test[:3], dtype=tf.float32))
print("Raw SavedModel prediction shape:", y_pred_raw.shape)

# 2. As a Keras model
keras_model = keras.models.load_model(model_path)
print("Keras reloaded model accuracy:", keras_model.evaluate(X_test, y_test)[1])

## Convert to TFLite (for mobile/embedded) with post-training quantization

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # enables post-training quantization
tflite_model = converter.convert()

with open("converted_model.tflite", "wb") as f:
    f.write(tflite_model)

original_size = sum(os.path.getsize(os.path.join(dp, f))
                     for dp, _, fs in os.walk(model_path) for f in fs)
tflite_size = os.path.getsize("converted_model.tflite")
print(f"SavedModel size: {original_size/1024:.1f} KB   TFLite size: {tflite_size/1024:.1f} KB")

## Data parallelism across multiple GPUs (pattern — needs actual multi-GPU hardware to see the speedup)

In [ ]:
distribution = tf.distribute.MirroredStrategy()
print("Number of devices:", distribution.num_replicas_in_sync)

with distribution.scope():
    mirrored_model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    mirrored_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

mirrored_model.fit(X_train, y_train, epochs=2, batch_size=100, validation_split=0.1)

## ✍️ Practice
1. Inspect the SavedModel above with `saved_model_cli show --dir ... --all` — what are the input/output tensor names and shapes?
2. Why does bundling preprocessing inside an exported model reduce training/serving skew risk?
3. Compare file size and test accuracy of the model before and after TFLite quantization above.
4. Debug: adding a 2nd GPU only gives a 1.3x speedup instead of ~2x — what's the likely explanation? Is this surprising?

In [ ]:
# Your code / notes here